# **ETL of Our World in Data: Population**

## Objectives
* read in Population data
* clean the data, apply the iso3 country code
* append to the cleaned CO2 data/csv

## Inputs

* data/raw/population-with-un-projections/population-with-un-projections.csv
* data/processed/01a_df_owid_co2_country.csv

## Outputs

* data/processed/01b_df_owid_co2_pop_country.csv
* data/processed/01b_df_owid_co2_world.csv??



---

# Set Up directories and Import Necessary Libraries

In [100]:
# System and OS related tasks
import sys
import os

# Add the project root to Python path
project_root = os.path.abspath('..')
sys.path.insert(0, project_root)

# path to directories
raw_dir = '../data/raw'
processed_dir = '../data/processed'

In [101]:
# Get the necessary libraries
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

sns.set_style('whitegrid') # sets a white background with grid lines 

# 1.0 OWID Population Data

In [102]:
data_columns = ["country", "country_code", "year", "pop_actual", "pop_projected"]
column_dtypes = {
    "country": str
    , "country_code": str
    , "year": int
    , "pop_actual" : float
    , "pop_projected" : float}

In [103]:
file_dir_name = "/population-with-un-projections/population-with-un-projections.csv"

df_owid_pop = pd.read_csv(
    f"{raw_dir}/{file_dir_name}",
    header=0,
    names=data_columns,
    dtype=column_dtypes
    )

print(f"Total number of rows: {len(df_owid_pop)}")

Total number of rows: 39562


In [104]:
df_owid_pop

,country,country_code,year,pop_actual,pop_projected
0,Afghanistan,AFG,1950,7776180.0,NaN
1,Afghanistan,AFG,1951,7879343.0,NaN
2,Afghanistan,AFG,1952,7987784.0,NaN
3,Afghanistan,AFG,1953,8096703.0,NaN
4,Afghanistan,AFG,1954,8207954.0,NaN
...,...,...,...,...,...
39557,Zimbabwe,ZWE,2096,NaN,36840484.0
39558,Zimbabwe,ZWE,2097,NaN,36932280.0
39559,Zimbabwe,ZWE,2098,NaN,37019885.0
39560,Zimbabwe,ZWE,2099,NaN,37096555.0


In [105]:
# look at the unique country codes
unique_codes = df_owid_pop['country_code'].dropna().unique()
print(f"Total unique country codes: {len(unique_codes)}")

Total unique country codes: 254


In [106]:
# add the iso country code 
import country_converter as coco
cc = coco.CountryConverter()

df_owid_pop["country_code_iso3"] = cc.pandas_convert(
    series=df_owid_pop['country_code'],
    to='ISO3'
)


OWID_AFR not found in regex
UN_AFR not found in regex
nan not found in ISO3
OWID_ASI not found in regex
UN_ASI not found in regex
OWID_EUR not found in regex
UN_EUR not found in regex
OWID_HIC not found in regex
OWID_KOS not found in regex
UN_LAC not found in regex
OWID_LIC not found in regex
OWID_LMC not found in regex
OWID_NAM not found in regex
UN_NAM not found in regex
OWID_OCE not found in regex
UN_OCE not found in regex
OWID_SAM not found in regex
OWID_UMC not found in regex
OWID_WRL not found in regex


📌 We already know about "OWID_KOS" and "OWID_WRL" and so these other `country_code` values look like aggregations rows which we do not want in our main dataset:

In [107]:
country_code_mapping = {
    "OWID_AFR"              # Africa
    , "OWID_ASI"            # Asia 
    , "OWID_EUR"            # Europe  
    , "OWID_HIC"            # High Income countries  
    , "OWID_LIC"            # Low Income countries   
    , "OWID_LMC"            # Low Middle Income countries   
    , "OWID_NAM"            # North America
    , "OWID_OCE"            # Oceania  
    , "OWID_SAM"            # South America  
    , "OWID_UMC"            # Upper Middle Income countries   
    , "OWID_WRL"            # WORLD
    , "UN_AFR"              # Africa UN Version
    , "UN_ASI"              # Asia UN Version
    , "UN_EUR"              # Europe UN Version
    , "UN_LAC"              # Latin America & Caribbean UN Version 
    , "UN_NAM"              # North America UN Version 
    , "UN_OCE"              # Oceania UN Version 
}

📌 Check the missing values in `country_code` and the data looks like aggregations too.

In [108]:
# find the missing values in country_code
nan_rows =df_owid_pop[df_owid_pop['country_code'].isna()]

nan_rows['country'].value_counts()

country
Americas (UN)                                                  151
Land-locked developing countries (LLDC)                        151
Least developed countries                                      151
Less developed regions                                         151
Less developed regions, excluding China                        151
Less developed regions, excluding least developed countries    151
More developed regions                                         151
Small island developing states (SIDS)                          151
Name: count, dtype: int64

In [109]:
nan_rows.head(10)

,country,country_code,year,pop_actual,pop_projected,country_code_iso3
906,Americas (UN),NaN,1950,335791496.0,NaN,not found
907,Americas (UN),NaN,1951,342809279.0,NaN,not found
908,Americas (UN),NaN,1952,350086867.0,NaN,not found
909,Americas (UN),NaN,1953,357599609.0,NaN,not found
910,Americas (UN),NaN,1954,365377091.0,NaN,not found
911,Americas (UN),NaN,1955,373407276.0,NaN,not found
912,Americas (UN),NaN,1956,381692163.0,NaN,not found
913,Americas (UN),NaN,1957,390249080.0,NaN,not found
914,Americas (UN),NaN,1958,398992351.0,NaN,not found
915,Americas (UN),NaN,1959,408077039.0,NaN,not found


## 2.1 Country Code "OWID_KOS"

📌 The `country_code` column in the OWID data with the value `OWID_KOS` refers to the country Kosovo. 

Kosovo is not yet a member of the United Nations and as such the Country_Converter package [on Kosovo](https://github.com/vincentarelbundock/countrycode/issues/305) does know what ISO3 code to assign to it.

Many commercial datasets and the World's bank has provisionally used `XKX` for Kosovo. We will manually use that as the country_code instead.

In [110]:
kos_rows = df_owid_pop[df_owid_pop['country_code'] == 'OWID_KOS']

kos_rows.head(5)

,country,country_code,year,pop_actual,pop_projected,country_code_iso3
17969,Kosovo,OWID_KOS,1950,778478.0,NaN,not found
17970,Kosovo,OWID_KOS,1951,794334.0,NaN,not found
17971,Kosovo,OWID_KOS,1952,811028.0,NaN,not found
17972,Kosovo,OWID_KOS,1953,828803.0,NaN,not found
17973,Kosovo,OWID_KOS,1954,847888.0,NaN,not found


In [111]:
# find the rows with country_code = "OWID_KOS" and assign the value "XKX" to the country_code_iso3

df_owid_pop["country_code_iso3"] = df_owid_pop["country_code"].replace("OWID_KOS", "XKX")


kos_rows = df_owid_pop[df_owid_pop['country_code'] == 'OWID_KOS']
kos_rows.head(5)

,country,country_code,year,pop_actual,pop_projected,country_code_iso3
17969,Kosovo,OWID_KOS,1950,778478.0,NaN,XKX
17970,Kosovo,OWID_KOS,1951,794334.0,NaN,XKX
17971,Kosovo,OWID_KOS,1952,811028.0,NaN,XKX
17972,Kosovo,OWID_KOS,1953,828803.0,NaN,XKX
17973,Kosovo,OWID_KOS,1954,847888.0,NaN,XKX


## 2.2 Aggregations Country Code values and nan

📌 The `country_code` column in the OWID population data has the following values that refes to the aggregation of populations at different granularity. Any rows with nan in the same country is also an aggregations.

In [112]:
country_code_mapping

{'OWID_AFR',
 'OWID_ASI',
 'OWID_EUR',
 'OWID_HIC',
 'OWID_LIC',
 'OWID_LMC',
 'OWID_NAM',
 'OWID_OCE',
 'OWID_SAM',
 'OWID_UMC',
 'OWID_WRL',
 'UN_AFR',
 'UN_ASI',
 'UN_EUR',
 'UN_LAC',
 'UN_NAM',
 'UN_OCE'}

In [113]:
# create a  query mask for identifying rows with with aggregation data or is nan
exclude_mask = (df_owid_pop["country_code"].isin(country_code_mapping) ) | (df_owid_pop["country_code"].isna())

### 2.2.1 Get and Check the World aggregation data

In [114]:
# get a copy of the df_owid_co2 by including all the unwanted rows
df_owid_pop_world = df_owid_pop[exclude_mask].copy()

In [115]:
num_aggre_rows = (df_owid_pop_world["country_code"].isin(country_code_mapping)).sum()
num_nan_rows = df_owid_pop_world["country_code"].isna().sum()

print(f"Total ros in df_owid_pop: {len(df_owid_pop)}")
print(f"OWID_WRL rows: {num_aggre_rows}")
print(f"NaN rows: {num_nan_rows}")
print(f"Total rows to be excluded: {num_aggre_rows + num_nan_rows}")
print(f"Country Specific rows left: {len(df_owid_pop) - num_aggre_rows - num_nan_rows}")

Total ros in df_owid_pop: 39562
OWID_WRL rows: 2567
NaN rows: 1208
Total rows to be excluded: 3775
Country Specific rows left: 35787


📌 Export to csv

In [116]:
df_owid_pop_world.to_csv(f'{processed_dir}/01b_df_owid_pop_world.csv', index=False)
print(f"Exported: {processed_dir}/01b_df_owid_pop_world.csv")

Exported: ../data/processed/01b_df_owid_pop_world.csv


### 2.2.2 Get and Check the country-year dataset

In [117]:
# get a copy of the df_owid_co2 by excluding all the unwanted rows
df_owid_pop_country = df_owid_pop[~exclude_mask].copy()

In [118]:
print(f"Rows with OWID_WRL: {(df_owid_pop_world["country_code"] == "OWID_WRL").sum()}")
print(f"Rows with missing value: {df_owid_pop_world["country_code"].isna().sum()}")
print(f"Country Specific rows left: {len(df_owid_pop_country)}")

Rows with OWID_WRL: 151
Rows with missing value: 1208
Country Specific rows left: 35787


In [119]:
check_kosovo = df_owid_pop_country[df_owid_pop_country["country"] == "Kosovo"]

check_kosovo.head(2)

,country,country_code,year,pop_actual,pop_projected,country_code_iso3
17969,Kosovo,OWID_KOS,1950,778478.0,NaN,XKX
17970,Kosovo,OWID_KOS,1951,794334.0,NaN,XKX


📌 Export to csv

In [120]:
df_owid_pop_country.to_csv(f'{processed_dir}/01b_df_owid_pop_country.csv', index=False)
print(f"Exported: {processed_dir}/01b_df_owid_pop_country.csv")

Exported: ../data/processed/01b_df_owid_pop_country.csv


# 2.3 Merging the population columns

📌 The columns `pop_actual` and `pop_projected` contains actual population values for each country-year and the projected population values.

For the main dataset, we want to 1 single column for population.

In [121]:
df_owid_pop_country["year"].describe()

count    35787.000000
mean      2025.000000
std         43.589598
min       1950.000000
25%       1987.000000
50%       2025.000000
75%       2063.000000
max       2100.000000
Name: year, dtype: float64

📌 The max year is year 2100. Lets look at from which year does the projected values starts. 

In [122]:
print(f"First projected year:{df_owid_pop_country[df_owid_pop_country['pop_projected'].notna()]['year'].min()}")

First projected year:2024


📌 The population values are projected from year 2024. We will only take the data from 1950 to 2024.

In [123]:
df_owid_pop_country_1970_2024 = df_owid_pop_country[
    (df_owid_pop_country["year"] >= 1970) &
    (df_owid_pop_country["year"] <= 2024)
]

In [124]:
df_owid_pop_country_1970_2024["year"].describe()

count    13035.000000
mean      1997.000000
std         15.875117
min       1970.000000
25%       1983.000000
50%       1997.000000
75%       2011.000000
max       2024.000000
Name: year, dtype: float64

📌 Verfiy the population columns data for year 2024:

In [125]:
df_2024_only = df_owid_pop_country_1970_2024[df_owid_pop_country_1970_2024["year"] == 2024]

# get rows with pop_actual = nan for year 2024 
df_2024_only_actual_nan = df_2024_only["pop_actual"].isna().all
# get rows with pop_project not = nan for year 2024 
df_2024_only_projected_not_nan = df_2024_only["pop_projected"].notna().all

# if a row is in df_2024_only_actual_nan and also in df_2024_only_projected_not_nan then we got the correct data
if df_2024_only_actual_nan and df_2024_only_projected_not_nan :
    print("All the 2024 rows have projected values only")
else:
    print("oops! Data error!")    

All the 2024 rows have projected values only


📌 Merge pop_actual and pop_projected into a new population column such that 
* for years < 2024, the population column will take pop_actual and 
* for year 2024 the population column will take pop_projected

In [126]:
df_owid_pop_country_1970_2024["population"] = df_owid_pop_country_1970_2024.apply(
    lambda row: row["pop_projected"] if row["year"] == 2024 else row["pop_actual"],
    axis=1 # for each row
)

print(f"Total df_owid_pop_country_1950_2024 rows: {len(df_owid_pop_country_1970_2024)}")
print(f"Non null population rows: {df_owid_pop_country_1970_2024["population"].notna().sum()}")
print(f"Null population rows: {df_owid_pop_country_1970_2024["population"].isna().sum()}")

Total df_owid_pop_country_1950_2024 rows: 13035
Non null population rows: 13035
Null population rows: 0


📌 Verify by country whether there are 
* missing year.
* duplicate country-year rows

In [127]:
#check for duplcates country-year rows
country_year_counts = df_owid_pop_country_1970_2024.groupby(
    ["country", "year"]
).size()

print(f"Number of duplicated country-year: {len(country_year_counts[country_year_counts > 1])}")

Number of duplicated country-year: 0


In [128]:
country_counts = df_owid_pop_country_1970_2024['country'].value_counts()

# Check if all have 2024 - 1970 + 1 = 55 rows
all_have = (country_counts == 55).all()
print(f"\nAll countries have 55 years: {all_have}")


All countries have 55 years: True


# 2.4 Export population data frame to csv

In [129]:
df_owid_pop_country_1970_2024 = df_owid_pop_country_1970_2024.drop(
    columns=['pop_actual', 'pop_projected']
)

In [130]:
df_owid_pop_country_1970_2024.to_csv(f'{processed_dir}/01b_df_owid_pop_country_1970_2024.csv', index=False)
print(f"Exported: {processed_dir}/01b_df_owid_pop_country_1970_2024.csv")

Exported: ../data/processed/01b_df_owid_pop_country_1970_2024.csv


---

# 3.0 Merge the CO<sub>2</sub> data with Population Data

📌 Read in the CO2 csv data.

In [131]:
data_columns = [
        "country"
        , "country_code"
        , "year"
        , "co2_pc"
        , "country_code_iso3"
        , "country_name_iso3"]

column_dtypes = {
    "country": str
    , "country_code": str
    , "year": int
    , "co2_pc" : float
    , "country_code_iso3" : str
    , "country_name_iso3": str}

file_dir_name = "01a_df_owid_co2_country_aft1969.csv"

df_owid_co2_country = pd.read_csv(
    f"{processed_dir}/{file_dir_name}",
    header=0,
    names=data_columns,
    dtype=column_dtypes
    )

df_owid_co2_country.head(3)

,country,country_code,year,co2_pc,country_code_iso3,country_name_iso3
0,Afghanistan,AFG,1970,0.147952,AFG,Afghanistan
1,Afghanistan,AFG,1971,0.163694,AFG,Afghanistan
2,Afghanistan,AFG,1972,0.129103,AFG,Afghanistan


In [132]:
len(df_owid_co2_country)

11479

📌 Merge CO<sub>2</sub> data with population data keeping all the rows in CO<sub>2</sub> data.

In [135]:
df_owid_co2_pop_1970_2024 = df_owid_co2_country.merge(
    df_owid_pop_country_1970_2024[["country_code_iso3", "year", "population"]]
    , on=["country_code_iso3", "year"]
    , how="left"
) 

print("="*80)
print("LEFT MERGE RESULTS:")
print("="*80)
print(f"Original CO2 rows: {len(df_owid_co2_country)}")
print(f"Merged rows: {len(df_owid_co2_pop_1970_2024) }")
print(f"Rows with population data: {df_owid_co2_pop_1970_2024 ['population'].notna().sum()}")
print(f"Rows missing population: {df_owid_co2_pop_1970_2024 ['population'].isna().sum()}")
print(f"Population not nan: {df_owid_co2_pop_1970_2024 ['population'].notna().sum()}")

LEFT MERGE RESULTS:
Original CO2 rows: 11479
Merged rows: 11479
Rows with population data: 11479
Rows missing population: 0
Population not nan: 11479


In [136]:
df_owid_co2_pop_1970_2024.head(5)

,country,country_code,year,co2_pc,country_code_iso3,country_name_iso3,population
0,Afghanistan,AFG,1970,0.147952,AFG,Afghanistan,11290134.0
1,Afghanistan,AFG,1971,0.163694,AFG,Afghanistan,11567672.0
2,Afghanistan,AFG,1972,0.129103,AFG,Afghanistan,11853698.0
3,Afghanistan,AFG,1973,0.134517,AFG,Afghanistan,12158000.0
4,Afghanistan,AFG,1974,0.153431,AFG,Afghanistan,12469127.0


📌 Export CO<sub>2</sub> and population data to csv.

In [137]:
df_owid_co2_pop_1970_2024.to_csv(f'{processed_dir}/01b_df_owid_co2_pop_1970_2024.csv', index=False)
print(f"Exported: {processed_dir}/01b_owid_df_co2_pop_1970_2024.csv")

Exported: ../data/processed/01b_owid_df_co2_pop_1970_2024.csv


---